In [ ]:
# ==============================================================================
# CÉLULA 1: Instalação de Dependências e Configuração de Ambiente
# ==============================================================================

# Instalação do SDK da OpenAI (usado para chamar a API do OpenRouter) e utilitários
!pip install -q openai pandas tabulate

import sqlite3
import json
import re
from typing import Dict, Any, Tuple, Optional
import pandas as pd
from tabulate import tabulate
from openai import OpenAI

print("Dependências instaladas e importadas com sucesso!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copia do Drive para o ambiente local (ajuste o caminho da pasta se necessário)
!cp "/content/drive/MyDrive/cinerocket.db" "cinerocket.db"

In [ ]:
# ==============================================================================
# CÉLULA 2: Verificação do Arquivo e Extração do Schema (Camada Gold)
# ==============================================================================

import os
import sqlite3
import pandas as pd
from tabulate import tabulate

DB_PATH = "cinerocket.db"

# Verifica se o arquivo existe e tem tamanho > 0
if os.path.exists(DB_PATH):
    size_mb = os.path.getsize(DB_PATH) / (1024 * 1024)
    print(f" Arquivo '{DB_PATH}' encontrado! Tamanho: {size_mb:.2f} MB\n")
else:
    raise FileNotFoundError(f"Arquivo {DB_PATH} não encontrado na raiz!")

def inspect_database_schema(db_path: str):
    """
    Inspeciona o banco SQLite, extrai o DDL/colunas de todas as tabelas
    e retorna uma representação textual formatada para alimentar o prompt do agente.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Lista tabelas
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")
    tables = [row[0] for row in cursor.fetchall()]

    schema_summary = []
    schema_dict = {}

    print(f" Tabelas encontradas no banco ({len(tables)}): {', '.join(tables)}\n")
    print("=" * 80)

    for table in tables:
        cursor.execute(f"PRAGMA table_info({table});")
        columns = cursor.fetchall()
        col_descriptions = [f"{col[1]} ({col[2]})" for col in columns]
        schema_dict[table] = [col[1] for col in columns]

        df_sample = pd.read_sql_query(f"SELECT * FROM {table} LIMIT 2;", conn)

        table_doc = f"Table: {table}\nColumns: {', '.join(col_descriptions)}"
        schema_summary.append(table_doc)

        print(f"Tabela: {table}")
        print(f"Colunas: {', '.join(col_descriptions)}")
        if not df_sample.empty:
            print("Amostra:")
            print(tabulate(df_sample, headers='keys', tablefmt='psql', showindex=False))
        else:
            print("(Tabela vazia)")
        print("-" * 80)

    conn.close()
    return "\n\n".join(schema_summary), schema_dict

GOLD_SCHEMA_PROMPT, DB_COLUMNS_DICT = inspect_database_schema(DB_PATH)

In [ ]:
# ==============================================================================
# CÉLULA 3: Configuração da Chave e Teste com Modelo :free Válido
# ==============================================================================

import requests
from getpass import getpass
from openai import OpenAI

# Se gerou uma nova chave sem limite, insira-a aqui:
OPENROUTER_API_KEY = getpass("Insira sua chave OpenRouter com limite liberado:")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "https://colab.research.google.com",
        "X-Title": "CineData Analytics GenAI Agent",
    }
)

def get_pure_free_models():
    """Retorna apenas modelos que contenham explicitamente :free no slug."""
    url = "https://openrouter.ai/api/v1/models"
    headers = {"Authorization": f"Bearer {OPENROUTER_API_KEY}"}
    resp = requests.get(url, headers=headers)
    if resp.status_code != 200:
        return []
    models = resp.json().get("data", [])
    return [m["id"] for m in models if m["id"].endswith(":free")]

free_models = get_pure_free_models()
print(f" Modelos com sufixo :free disponíveis ({len(free_models)}):")
for m in free_models:
    print(f" - {m}")

# Testa em ordem até achar um que responda com sucesso
ACTIVE_MODEL = None
for candidate in free_models:
    try:
        print(f"Testando {candidate}...")
        test_resp = client.chat.completions.create(
            model=candidate,
            messages=[{"role": "user", "content": "Diga apenas: OK"}],
            max_tokens=10
        )
        ACTIVE_MODEL = candidate
        print(f" Conexão validada com sucesso com: {ACTIVE_MODEL}")
        print(f" Retorno: {test_resp.choices[0].message.content.strip()}")
        break
    except Exception as e:
        print(f"    Falha ao testar {candidate}: {e}")

if not ACTIVE_MODEL:
    print(" Nenhum modelo gratuito respondeu. Verifique se o limite da chave foi atualizado no OpenRouter.")

In [ ]:
# ==============================================================================
# CÉLULA 4: Guardrails de Segurança, Validação SQL e Execução Local com Cache
# ==============================================================================

import re
import sqlite3
import pandas as pd
from typing import Dict, Any, Tuple, Optional

# Cache em memória para poupar requisições da cota diária
SQL_AGENT_CACHE: Dict[str, Dict[str, Any]] = {}

# Palavras-chave proibidas para garantir que o agente opere exclusivamente como Read-Only
DANGEROUS_SQL_KEYWORDS = [
    r"\bDROP\b", r"\bDELETE\b", r"\bUPDATE\b", r"\bINSERT\b",
    r"\bALTER\b", r"\bTRUNCATE\b", r"\bEXEC\b", r"\bCREATE\b",
    r"\bREPLACE\b", r"\bATTACH\b", r"\bDETACH\b", r"\bPRAGMA\b"
]

def sanitize_and_validate_sql(raw_sql: str) -> Tuple[bool, str]:
    """
    Remove blocos de markdown e espaços extras, além de validar se a query
    é estritamente uma operação de leitura segura (SELECT / WITH).
    """
    cleaned_sql = raw_sql.strip()

    # Remove marcação de markdown se o modelo envolver com ```sql ... ```
    cleaned_sql = re.sub(r"^```(?:sql)?\s*", "", cleaned_sql, flags=re.IGNORECASE)
    cleaned_sql = re.sub(r"\s*```$", "", cleaned_sql)
    cleaned_sql = cleaned_sql.strip().rstrip(";") + ";"

    # 1. Verifica comandos proibidos de mutação
    for pattern in DANGEROUS_SQL_KEYWORDS:
        if re.search(pattern, cleaned_sql, re.IGNORECASE):
            return False, f"Violação de Guardrail: Comando SQL proibido detectado ({pattern}). Apenas leitura é permitida."

    # 2. Garante que comece com SELECT ou WITH (para CTEs)
    if not re.match(r"^(SELECT|WITH)\b", cleaned_sql, re.IGNORECASE):
        return False, "Violação de Guardrail: A consulta deve iniciar estritamente com SELECT ou WITH."

    return True, cleaned_sql

def execute_gold_query(sql_query: str, db_path: str = "cinerocket.db") -> Tuple[bool, Any]:
    """
    Executa a query sanitizada no SQLite local e retorna um Pandas DataFrame
    ou a mensagem de erro da engine caso haja falha de sintaxe.
    """
    is_valid, sanitized_or_err = sanitize_and_validate_sql(sql_query)
    if not is_valid:
        return False, sanitized_or_err

    try:
        conn = sqlite3.connect(db_path)
        df_result = pd.read_sql_query(sanitized_or_err, conn)
        conn.close()
        return True, df_result
    except Exception as e:
        return False, f"Erro de execução SQLite: {str(e)}"

# Teste simples do Guardrail
safe_test, safe_sql = sanitize_and_validate_sql("```sql\nSELECT titulo, ano_lancamento FROM dim_movies LIMIT 3;\n```")
unsafe_test, unsafe_msg = sanitize_and_validate_sql("DROP TABLE dim_movies;")

print(f" Teste SQL Seguro: {'Válido' if safe_test else 'Bloqueado'} -> {safe_sql}")
print(f" Teste SQL Perigoso: {'Válido' if unsafe_test else 'Bloqueado'} -> {unsafe_msg}")

# Teste de execução no banco
success, test_df = execute_gold_query(safe_sql)
if success:
    print("\n Execução de leitura no SQLite testada com sucesso:")
    print(test_df)
else:
    print(f" Erro na execução: {test_df}")

In [ ]:
# ==============================================================================
# CÉLULA 5: Engenharia de Prompts, Pipeline Text-to-SQL e Síntese de Resposta
# ==============================================================================

import re
import json
import pandas as pd
from tabulate import tabulate

# Prompt do Sistema contendo regras de negócio, schema e boas práticas SQLite
SYSTEM_SQL_PROMPT = f"""Você é um Engenheiro de Analytics e Especialista em SQL para o Data Lakehouse CineData Analytics.
Sua missão é converter perguntas em linguagem natural de analistas de negócio em consultas SQLite 3 precisas e eficientes.

### ESQUEMA DO BANCO DE DADOS (Camada Gold):
{GOLD_SCHEMA_PROMPT}

### DIRETRIZES E REGRAS DE NEGÓCIO MANDATÓRIAS:
1. Retorne ESTRITAMENTE a consulta SQL dentro de um bloco de código markdown com sql. Não inclua texto introdutório ou saudações.
2. Apenas operações de LEITURA (SELECT ou WITH) são permitidas.
3. Tratamento de Valores e Métricas Financeiras:
   - "Receita" ou "Bilheteria" ou "Faturamento": use a coluna `receita_brl` (ou `receita_usd` se solicitado explicitamente em dólar).
   - "Orçamento": use a coluna `orcamento_brl` (ou `orcamento_usd`).
   - "Lucro": use `lucro_brl` ou `(receita_brl - orcamento_brl)`.
   - "Margem de Lucro": use `(CAST((receita_brl - orcamento_brl) AS REAL) / receita_brl)` ou `(CAST(lucro_brl AS REAL) / receita_brl)`.
   - Sempre filtre `receita_brl IS NOT NULL AND receita_brl > 0` e `orcamento_brl IS NOT NULL AND orcamento_brl > 0` para cálculos de lucro ou margem.
4. Relações e Junções:
   - Para associar filmes a gêneros: `dim_movies m JOIN bridge_movie_genre bmg ON m.sk_movie_id = bmg.sk_movie_id JOIN dim_genres g ON bmg.sk_genre_id = g.sk_genre_id`.
   - Para associar filmes a atores/diretores: `dim_movies m JOIN bridge_movie_person bmp ON m.sk_movie_id = bmp.sk_movie_id JOIN dim_people p ON bmp.sk_person_id = p.sk_person_id`. Filtre `p.tipo_pessoa = 'Diretor'` ou `p.tipo_pessoa = 'Ator'` conforme solicitado.
   - Para associar produtoras: `bridge_movie_company` e `dim_companies`.
   - Para avaliações agregadas de usuários: `dim_reviews` (`nota_media_usuarios`, `qtd_avaliacoes_usuarios`).
   - Para avaliações TMDB/IMDb e finanças: `fact_movies_performance`.
5. Se o usuário pedir um ranking ou "Top N", sempre use `ORDER BY ... DESC LIMIT N`.
6. Use nomes amigáveis de colunas com aliases (`AS`).
"""

def extract_sql_from_response(raw_text: str) -> str:
    """
    Extrai com precisão o bloco SQL da resposta do modelo,
    ignorando blocos de raciocínio (thinking) ou textos explicativos.
    """
    sql_blocks = re.findall(r"```(?:sql)?\s*([\s\S]*?)\s*```", raw_text, re.IGNORECASE)
    if sql_blocks:
        return sql_blocks[-1].strip()

    # Fallback caso o modelo não use crases: busca a partir do SELECT/WITH
    match = re.search(r"\b(SELECT|WITH)\b[\s\S]+", raw_text, re.IGNORECASE)
    if match:
        return match.group(0).strip()

    return raw_text.strip()

def generate_sql(question: str, model_name: str = ACTIVE_MODEL) -> str:
    """Invoca o LLM para traduzir a pergunta em linguagem natural para SQL."""
    messages = [
        {"role": "system", "content": SYSTEM_SQL_PROMPT},
        {"role": "user", "content": f"Pergunta do usuário: {question}\nGere a consulta SQL SQLite correspondente."}
    ]
    response = client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0.0
    )
    raw_content = response.choices[0].message.content or ""
    return extract_sql_from_response(raw_content)

def synthesize_answer(question: str, sql_query: str, df_data: pd.DataFrame, model_name: str = ACTIVE_MODEL) -> str:
    """Recebe os dados obtidos no SQLite e gera uma análise sintetizada em linguagem natural."""
    if df_data.empty:
        return "Nenhum resultado foi retornado do banco para os critérios solicitados."

    sample_text = df_data.head(10).to_string(index=False)

    prompt = (
        "Você é o assistente inteligente da CineData Analytics.\n"
        "Apresente uma resposta executiva e objetiva para a pergunta do usuário com base nos dados reais retornados do banco.\n\n"
        f"Pergunta: {question}\n"
        f"Consulta SQL executada:\n{sql_query}\n\n"
        f"Dados obtidos:\n{sample_text}\n\n"
        "Diretrizes:\n"
        "- Destaque valores principais em negrito.\n"
        "- Formate valores monetários em R$ de forma clara.\n"
        "- Seja direto e não invente dados além dos fornecidos."
    )

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    return response.choices[0].message.content or "Não foi possível gerar a síntese."

def ask_cinedata(question: str, verbose: bool = True) -> dict:
    """
    Pipeline principal do Agente CineData:
    1. Consulta Cache
    2. Geração Text-to-SQL
    3. Validação e Execução
    4. Auto-correção em caso de erro sintático (1 tentativa)
    5. Síntese final dos dados
    """
    norm_q = question.strip().lower()

    # 1. Checa Cache
    if norm_q in SQL_AGENT_CACHE:
        if verbose:
            print(" [Cache Hit] Resposta recuperada instantaneamente da memória!")
        return SQL_AGENT_CACHE[norm_q]

    if verbose:
        print(f" Pergunta: \"{question}\"")
        print(" Gerando consulta SQL...")

    sql_query = generate_sql(question)

    # 2. Executa consulta
    success, result = execute_gold_query(sql_query)

    # 3. Retry / Self-Healing se houver erro sintático
    if not success:
        if verbose:
            print(f" Erro na execução inicial: {result}")
            print(" Solicitando auto-correção do SQL...")
        retry_prompt = [
            {"role": "system", "content": SYSTEM_SQL_PROMPT},
            {"role": "user", "content": f"A consulta:\n{sql_query}\nGerou o seguinte erro SQLite: {result}\nCorrija a consulta mantendo as regras."}
        ]
        retry_resp = client.chat.completions.create(model=ACTIVE_MODEL, messages=retry_prompt, temperature=0.0)
        sql_query = extract_sql_from_response(retry_resp.choices[0].message.content or "")
        success, result = execute_gold_query(sql_query)

    if not success:
        return {
            "status": "error",
            "question": question,
            "sql": sql_query,
            "error": result,
            "answer": f"Não foi possível executar a consulta. Erro: {result}"
        }

    df_data = result
    if verbose:
        print(" Consulta executada com sucesso! Gerando resposta final...")

    # 4. Síntese
    answer_text = synthesize_answer(question, sql_query, df_data)

    output = {
        "status": "success",
        "question": question,
        "sql": sql_query,
        "data": df_data,
        "answer": answer_text
    }

    SQL_AGENT_CACHE[norm_q] = output
    return output

print(" Pipeline e Agente Text-to-SQL configurados com sucesso!")

In [ ]:
# ==============================================================================
# CÉLULA 6: Bateria de Validação Prática das Perguntas de Negócio
# ==============================================================================

from tabulate import tabulate

def display_agent_result(result: dict):
    """Exibe o resultado do agente de forma limpa e estruturada."""
    print("=" * 80)
    print(f" Pergunta: {result['question']}")
    print("-" * 80)
    print(" SQL Gerado:")
    print(result.get('sql', 'Nenhum SQL gerado'))
    print("-" * 80)

    if result.get("status") == "success" and isinstance(result.get("data"), pd.DataFrame):
        print(" Tabela de Dados (Top Registros):")
        print(tabulate(result['data'].head(5), headers='keys', tablefmt='psql', showindex=False))
        print("-" * 80)
        print(" Resposta do Agente:")
        print(result.get('answer'))
    else:
        print(" Erro:")
        print(result.get('error') or result.get('answer'))
    print("=" * 80 + "\n")

# Lista de perguntas de teste representativas do enunciado
test_questions = [
    "Quais são os 5 filmes com maior receita em R$?",
    "Qual a quantidade de filmes por gênero?",
    "Quais são os diretores com maior nota média no IMDb, considerando apenas diretores com no mínimo 5 filmes?"
]

# Executa cada pergunta pelo agente
results = []
for q in test_questions:
    res = ask_cinedata(q)
    display_agent_result(res)
    results.append(res)

In [ ]:
# ==============================================================================
# CÉLULA 7: Formatação Numérica, Visualização Automática (DataViz) e Modo Interativo
# ==============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configuração visual padrão para relatórios executivos
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.size": 11, "figure.autolayout": True})

def format_currency_brl(val: float) -> str:
    """Converte números grandes em strings legíveis no padrão brasileiro."""
    if pd.isna(val) or val is None:
        return "N/A"
    abs_val = abs(val)
    if abs_val >= 1e9:
        return f"R$ {val / 1e9:.2f} bi"
    if abs_val >= 1e6:
        return f"R$ {val / 1e6:.2f} mi"
    if abs_val >= 1e3:
        return f"R$ {val / 1e3:.2f} mil"
    return f"R$ {val:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def render_chart_if_applicable(df: pd.DataFrame, title: str):
    """
    Analisa o DataFrame e gera automaticamente um gráfico adequado
    se houver uma coluna textual (rótulo) e ao menos uma coluna numérica.
    """
    if df.empty or len(df) < 2:
        return

    text_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if not text_cols or not num_cols:
        return

    label_col = text_cols[0]
    metric_col = num_cols[0]

    plot_df = df.head(10).copy()

    plt.figure(figsize=(9, 4.5))
    ax = sns.barplot(
        data=plot_df,
        x=metric_col,
        y=label_col,
        orient='h',
        color="#2b5c8f"
    )

    ax.set_title(title, fontsize=12, fontweight='bold', pad=12)
    ax.set_xlabel(metric_col.replace("_", " ").title())
    ax.set_ylabel("")

    # Formatação dos rótulos nas barras
    for p in ax.patches:
        width = p.get_width()
        if not np.isnan(width):
            if "receita" in metric_col or "lucro" in metric_col or "orcamento" in metric_col:
                label_txt = format_currency_brl(width)
            else:
                label_txt = f"{width:,.2f}".rstrip('0').rstrip('.')
            ax.annotate(
                label_txt,
                (width, p.get_y() + p.get_height() / 2.),
                va='center',
                ha='left',
                xytext=(5, 0),
                textcoords='offset points',
                fontsize=9
            )

    sns.despine(left=True, bottom=True)
    plt.show()

def ask_cinedata_interactive():
    """Interface de linha de comando no Colab para interação livre."""
    print("=" * 80)
    print(" CineData Analytics - Assistente GenAI Gold Layer")
    print("Digite sua pergunta de negócio (ou 'sair' para encerrar):")
    print("=" * 80)

    while True:
        user_input = input("\n Pergunta: ").strip()
        if user_input.lower() in ["sair", "exit", "quit"]:
            print("Encerrando sessão do assistente.")
            break
        if not user_input:
            continue

        res = ask_cinedata(user_input, verbose=False)
        print("\n" + "=" * 80)
        print(" SQL Executado:")
        print(res.get("sql", "N/A"))
        print("-" * 80)

        data = res.get("data")
        if isinstance(data, pd.DataFrame) and not data.empty:
            print(" Tabela (Primeiros registros):")
            print(tabulate(data.head(5), headers='keys', tablefmt='psql', showindex=False))
            print("-" * 80)

        print(" Resposta Executiva:")
        print(res.get("answer"))
        print("-" * 80)

        if isinstance(data, pd.DataFrame):
            render_chart_if_applicable(data, title=user_input)

print(" Módulo DataViz e Modo Interativo prontos para execução!")

In [ ]:
# ==============================================================================
# CÉLULA 8: Avaliação Automatizada por Categorias de Negócio (Golden Set)
# ==============================================================================

# Conjunto de perguntas cobrindo todas as categorias exigidas pelo enunciado
EVALUATION_SUITE = [
    {
        "categoria": "Bilheteria e Finanças",
        "pergunta": "Qual o lucro médio por gênero, considerando apenas filmes com receita informada?"
    },
    {
        "categoria": "Bilheteria e Finanças",
        "pergunta": "Quais os 5 filmes com maior margem de lucro, entre os que possuem receita e orçamento informados?"
    },
    {
        "categoria": "Popularidade e Engajamento",
        "pergunta": "Quais são os 5 filmes com maior divergência absoluta entre a nota TMDB e a nota IMDb?"
    },
    {
        "categoria": "Elenco e Equipe",
        "pergunta": "Qual a dupla ator e diretor que mais trabalhou junta e quantos filmes fizeram?"
    },
    {
        "categoria": "Gêneros e Produtoras",
        "pergunta": "Qual a produtora com maior lucro total em R$?"
    },
    {
        "categoria": "Avaliações dos Usuários",
        "pergunta": "Quais são os 5 filmes mais avaliados pelos usuários da plataforma?"
    }
]

eval_records = []

print(" Iniciando bateria de testes do Agente CineData Analytics...\n")

for item in EVALUATION_SUITE:
    cat = item["categoria"]
    q = item["pergunta"]
    print(f" Categoria: {cat}")
    print(f" Pergunta: {q}")

    res = ask_cinedata(q, verbose=False)
    status = res.get("status", "error")
    row_count = len(res["data"]) if status == "success" and isinstance(res.get("data"), pd.DataFrame) else 0

    eval_records.append({
        "Categoria": cat,
        "Pergunta": q,
        "Status": " OK" if status == "success" and row_count > 0 else " Falha",
        "Linhas Retornadas": row_count,
        "SQL Gerado": res.get("sql", "N/A")[:70] + "..." if res.get("sql") else "N/A"
    })

    print(f"Resultado: {'Sucesso' if status == 'success' else 'Falha'} ({row_count} registros retornados)")
    print("-" * 80)

# Tabela resumo da avaliação
df_eval = pd.DataFrame(eval_records)
print("\n RELATÓRIO CONSOLIDADO DE AVALIAÇÃO:")
print(tabulate(df_eval, headers='keys', tablefmt='psql', showindex=False))

In [ ]:
# ==============================================================================
# CÉLULA 9: TESTE UNITÁRIO
# ==============================================================================

# Recupera instantaneamente do cache e exibe a análise completa
res = ask_cinedata("Qual a dupla ator e diretor que mais trabalhou junta e quantos filmes fizeram?")
display_agent_result(res)